# ADL with the authors' official toolkit (smoke test: default + mix1-1p0)

Runs `science-of-finetuning/diffing-toolkit` (Minder et al.) unchanged, with two substitutions for Kaggle:

1. **vLLM is stubbed.** The toolkit imports vLLM at module level, but ADL itself generates with nnsight; the stub raises if anything
   actually tries to use vLLM.
2. **Graders run locally.** The OpenRouter GPT-5-mini/nano graders are replaced by Qwen3-4B-Instruct-2507 served through a small
   OpenAI-compatible endpoint, so the toolkit's own grader code and prompts are used unchanged.

Other overrides: fp16 (T4 has no bf16), both models on cuda:0, wandb off, `pipeline.mode=diffing` (no agent evaluation).


In [ ]:
%%bash
set -e
pip uninstall -y -q torchao vllm 2>/dev/null || true
git clone -q --depth 1 https://github.com/science-of-finetuning/diffing-toolkit.git /kaggle/working/dt
cd /kaggle/working/dt && git log -1 --format='toolkit commit %h %cs'
pip install -q "transformers==5.12.1" "nnsight==0.7.0" "nnterp==1.3.0" "peft==0.19.1" "accelerate==1.14.0" \
    "datasets==5.0.0" "hydra-core==1.3.3" "openai>=2.7.1" loguru python-dotenv coolname codenamize pathvalidate \
    torchnmf torchdr adjustText streamlit streamlit-tags st-annotated-text seaborn plotly hf-transfer gdown wandb \
    sentence-transformers fastapi uvicorn einops 2>&1 | tail -3
pip install -q --no-deps "git+https://github.com/Butanium/tiny-activation-dashboard.git" \
    "git+https://github.com/science-of-finetuning/crosscoder_learning.git" 2>&1 | tail -3
pip install -q --no-deps -e /kaggle/working/dt 2>&1 | tail -3
python -c "import torch, transformers, nnsight, nnterp; print('torch', torch.__version__, '| transformers', transformers.__version__, '| nnsight', nnsight.__version__, '| GPUs', torch.cuda.device_count())"


## vLLM stub and local grader server

In [ ]:
import os, subprocess, sys, time, json, textwrap, pathlib, urllib.request
STUB = pathlib.Path("/kaggle/working/stubs")
MODULES = ["vllm", "vllm.distributed", "vllm.inputs", "vllm.lora", "vllm.lora.request", "vllm.lora.peft_helper",
           "vllm.lora.models", "vllm.lora.lora_model", "vllm.logger", "vllm.entrypoints", "vllm.entrypoints.openai",
           "vllm.entrypoints.openai.protocol", "vllm.entrypoints.openai.api_server", "vllm.entrypoints.openai.engine",
           "vllm.entrypoints.openai.engine.protocol", "vllm.entrypoints.serve", "vllm.entrypoints.serve.lora",
           "vllm.entrypoints.serve.lora.protocol", "vllm.entrypoints.cli", "vllm.entrypoints.cli.main"]
BODY = textwrap.dedent("""
    import logging
    class _VLLMStub:
        def __init__(self, *a, **k):
            raise RuntimeError("vLLM is stubbed on this machine; this code path needs real vLLM")
    def init_logger(name):
        return logging.getLogger(name)
    def __getattr__(name):
        if name.startswith("__"):
            raise AttributeError(name)
        return _VLLMStub
""")
for m in MODULES:
    d = STUB.joinpath(*m.split("."))
    d.mkdir(parents=True, exist_ok=True)
    (d / "__init__.py").write_text(BODY)

import torch
N_GPU = torch.cuda.device_count()
GRADER_DEVICE = "cuda:1" if N_GPU > 1 else "cuda:0"
print("GPUs:", N_GPU, "| grader on", GRADER_DEVICE)

SERVER = textwrap.dedent("""
    import asyncio, os, time, uuid, torch, uvicorn
    from fastapi import FastAPI, Request
    from transformers import AutoTokenizer, AutoModelForCausalLM
    MODEL, DEVICE = "Qwen/Qwen3-4B-Instruct-2507", os.environ["GRADER_DEVICE"]
    MAX_BATCH, MAX_NEW = 16, 4096  # 1024 truncated the Patchscope grader before its answer line
    tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to(DEVICE).eval()
    app, queue = FastAPI(), asyncio.Queue()

    def flat(messages):  # the coherence grader sends OpenAI content-part lists; Qwen's template needs strings
        return [{**m, "content": m["content"] if isinstance(m["content"], str) else
                 "".join(p.get("text", "") for p in m["content"] if isinstance(p, dict))} for m in messages]

    def generate_batch(items):
        texts = [tok.apply_chat_template(flat(it["messages"]), add_generation_prompt=True, tokenize=False) for it in items]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        temps = [it["temperature"] for it in items]
        kw = dict(max_new_tokens=min(MAX_NEW, max(it["max_tokens"] for it in items)), pad_token_id=tok.pad_token_id)
        if all(t == 0 for t in temps):
            kw["do_sample"] = False
        else:
            kw.update(do_sample=True, temperature=max(0.1, sum(temps) / len(temps)), top_p=0.95)
        with torch.no_grad():
            out = model.generate(**enc, **kw)
        return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True) for o in out]

    async def worker():
        loop = asyncio.get_running_loop()
        while True:
            items = [await queue.get()]
            await asyncio.sleep(0.05)
            while not queue.empty() and len(items) < MAX_BATCH:
                items.append(queue.get_nowait())
            try:
                for it, text in zip(items, await loop.run_in_executor(None, generate_batch, items)):
                    it["fut"].set_result(text)
            except Exception as e:
                for it in items:
                    it["fut"].set_exception(e)

    @app.on_event("startup")
    async def startup():
        asyncio.create_task(worker())

    @app.get("/health")
    async def health():
        return {"ok": True}

    @app.post("/v1/chat/completions")
    async def chat(req: Request):
        body = await req.json()
        t = body.get("temperature")
        fut = asyncio.get_running_loop().create_future()
        await queue.put({"messages": body["messages"], "fut": fut, "temperature": 1.0 if t is None else float(t),
                         "max_tokens": int(body.get("max_tokens") or body.get("max_completion_tokens") or 512)})
        text = await fut
        return {"id": "chatcmpl-" + uuid.uuid4().hex, "object": "chat.completion", "created": int(time.time()),
                "model": body.get("model", "local"),
                "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
                "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}}

    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")
""")
pathlib.Path("/kaggle/working/grader_server.py").write_text(SERVER)
server_log = open("/kaggle/working/grader_server.log", "w")
server = subprocess.Popen([sys.executable, "/kaggle/working/grader_server.py"], stdout=server_log, stderr=subprocess.STDOUT,
                          env={**os.environ, "GRADER_DEVICE": GRADER_DEVICE})
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2); print("grader server up"); break
    except Exception:
        time.sleep(5)
else:
    print(open("/kaggle/working/grader_server.log").read()[-3000:]); raise SystemExit("grader server failed to start")

from openai import OpenAI
r = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local").chat.completions.create(
    model="local", messages=[{"role": "user", "content": "Reply with exactly: ANSWER: COHERENT"}], max_tokens=20)
print("grader check:", r.choices[0].message.content)


## Import check with the stub, then run ADL per variant

In [ ]:
DT = "/kaggle/working/dt"
ENV = {**os.environ, "PYTHONPATH": f"/kaggle/working/stubs:{DT}/src:" + os.environ.get("PYTHONPATH", ""),
       "OPENROUTER_API_KEY": "local", "WANDB_MODE": "disabled", "TOKENIZERS_PARALLELISM": "false"}
pathlib.Path(DT, "openrouter_api_key.txt").write_text("local")
chk = subprocess.run([sys.executable, "-c", "import diffing.pipeline.diffing_pipeline as p; print('toolkit import OK')"],
                     cwd=DT, env=ENV, capture_output=True, text=True)
print(chk.stdout, chk.stderr[-3000:])

LOCAL = "http://127.0.0.1:8000/v1"
OVERRIDES = [
    "pipeline.mode=diffing", "organism=cake_bake", "model=qwen3_1_7B", "diffing/method=activation_difference_lens",
    "infrastructure=runpod", "infrastructure.storage.base_dir=/kaggle/working/mo", "model.dtype=float16",
    "infrastructure.device_map.base=cuda:0", "infrastructure.device_map.finetuned=cuda:0", "wandb.enabled=false",
    "diffing.method.num_workers=2",
    "diffing.method.n=8",  # Fig. 8 uses positions 0-5; n=128 (positional study) needs ~20 GB RAM
] + [f"diffing.method.{g}.grader.{k}={v}" for g in ("auto_patch_scope", "steering", "token_relevance")
     for k, v in (("model_id", "local/qwen3-4b-instruct-2507"), ("base_url", LOCAL))]

VARIANTS = ["default", "mix1-1p0"]
timing = {}
for v in VARIANTS:
    t0 = time.time()
    log = pathlib.Path(f"/kaggle/working/adl_{v}.log")
    with open(log, "w") as f:
        rc = subprocess.run([sys.executable, "main.py", *OVERRIDES, f"organism_variant={v}"], cwd=DT, env=ENV,
                            stdout=f, stderr=subprocess.STDOUT).returncode
    timing[v] = round((time.time() - t0) / 60, 1)
    print(f"{v}: exit {rc}, {timing[v]} min")
    if rc != 0:
        print(log.read_text()[-6000:])
        break


## What the toolkit produced

In [ ]:
root = pathlib.Path("/kaggle/working/mo/diffing_results/qwen3_1_7B")
for p in sorted(root.rglob("*")):
    if p.is_file() and p.suffix in (".json", ".jsonl"):
        print(p.relative_to(root), p.stat().st_size)
for rel in sorted(root.rglob("relevance_*.json")):
    d = json.loads(rel.read_text())
    print("\n==", rel.relative_to(root)); print(json.dumps(d, indent=1)[:1500])
for thr in sorted(root.rglob("threshold.json")):
    print("\n==", thr.relative_to(root), thr.read_text()[:400])
print("\ntiming (min):", timing)
print("\ngrader server log tail:\n", open("/kaggle/working/grader_server.log").read()[-1500:])
